# C1: Inside the decision model (Strands hidden states, label-free)

**Question.** Does Strands 2B carry more usable information in its hidden states than in its own answer
probabilities, and where along the prompt does that information appear?

Strands reads `<state>` text `</state>` → question → `<options>` → `<answer>` causally. One label-free
forward per frozen feature-creation request (per text column and joint, identical deduplication) saves in
float16 the hidden state at the last token of the text, the last token of the question and the `<answer>`
token, at layer 12 and layer 24 (the pointer head's input). **C1a**: 3 positions × 2 layers × {per text column
(PCA32 per column), joint (PCA64)} = 12 arms; **C1b**: the `<answer>` layer-24 vector together with Strands'
own probabilities, per view = 2 arms. PCA is fitted on outer-training rows only; empty text stays missing.

Common protocol (Group B): seven fixed predictors, the no-text baseline, 18 datasets (9 classification,
9 regression), the same five outer folds, fixed headline metrics. Raw gains are reported within a task
family (ΔAUC; ΔR²) and never averaged across them; datasets are the unit and intervals resample datasets.
Strands' own probabilities (the B1 Strands arms) are carried by the C1 export on identical folds.
Design: `docs/EXPERIMENTS.md`, Group C; configuration `config/C/representations.yaml`.
The notebook reads only `output_JEVPFN/C/exp10_representations/results/group_c/c1_representations.zip`.

In [1]:
import sys
from pathlib import Path
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.notebook_display import display_json, display_local, display_table
from src.utils.notebook_report import begin_report, report_section, record, finish_report
from src.utils.provenance import provenance
from src.evaluation import group_c, group_c_views, learner_scores, study_export
from src.visualize import critical_difference, figures, group_c as gcfig, style
style.apply()
NOTEBOOK = "C/exp10_representations"
begin_report()
save = figures.FigureSaver(NOTEBOOK)
display_json("Notebook execution identity", provenance())
frames, receipt = group_c.read("C1")
display_local(study_export.describe(receipt))
record({k: receipt.get(k) for k in ("status", "path", "created_utc", "origin", "sha256", "bytes", "schema", "receipt_sha256")}, "Export identity")
WAITING = "Waiting for the C1 export (c1_representations.zip): nothing to show yet."
if frames:
    summary = group_c_views.overview(frames["dataset_gains"])
    contrasts = group_c_views.overview(frames["contrasts"])
    per_dataset = group_c_views.with_average(frames["dataset_gains"])
    names = group_c_views.dataset_names()

'Waiting for export: C:\\Users\\U0152019\\PhD Documents\\Projects\\5. JEVPFN\\JEVPFN\\output_JEVPFN\\C\\exp10_representations\\results\\group_c\\c1_representations.zip is absent. No measurements are shown.'

## 1. What is complete

Planned arms come from the benchmark plan (12 hidden-state arms and 2 vector + probability arms per
dataset and predictor); complete arms are those with five finite folds in the export. The per-dataset
head-reproduction contract (the released head on the saved float16 vectors must reproduce Strands'
probabilities within 0.005) is summarised here and tabulated in `general/4_study_audit`.

In [2]:
report_section("1. What is complete")
record(group_c.design_counts(), "Group C arms, configurations and C2 fits (plan, not results)")
if not frames:
    display_local(WAITING)
else:
    display_table("Planned versus complete five-fold arms per phase", group_c_views.coverage_summary(frames["coverage"]))
    display_table("Planned versus complete five-fold arms per predictor", frames["coverage"], preview_rows=14)
    contract = group_c_views.head_reproduction_contract(frames["extraction"])
    display_local(f"Hidden-state stores passing the head-reproduction contract: {int(contract.passed.sum())} of {len(contract)} datasets.")

'Waiting for the C1 export (c1_representations.zip): nothing to show yet.'

## 2. Summary: where along the prompt does usable information appear?

One figure: gain over no text along the prompt (end of text → end of question → `<answer>`) at layers 12
and 24, per view and task family, with Strands' own probabilities as the dotted reference and the C1b arm
as a star. The line is the equal-dataset mean of the per-dataset mean over the seven predictors; the band is
the range of the seven predictors.

In [3]:
report_section("2. Summary: gain along the prompt")
if not frames:
    display_local(WAITING)
else:
    gcfig.render(save, gcfig.position_path(summary, name="c1_position_path"))

'Waiting for the C1 export (c1_representations.zip): nothing to show yet.'

## 3. Detail: the approved comparisons, per dataset and per arm

* **Comparison forest**: all approved fold-paired C1 comparisons in one figure (the `<answer>` vector against
  and together with Strands' probabilities, each later position against the earlier one, layer 24 against
  layer 12); predictor mean with dataset-bootstrap 95% interval, dataset median and each predictor.
* **Per-dataset scatter**: the main C1 contrast dataset by dataset, vector gain against probability gain,
  with the identity line.
* **Supporting forests**: every C1 arm against no text, one figure per view.

In [4]:
report_section("3. Detail: comparisons and arms")
if not frames:
    display_local(WAITING)
else:
    gcfig.render(save, gcfig.c1_contrasts(contrasts, name="c1_contrasts"))
    gcfig.render(save, gcfig.c1_dataset_scatter(per_dataset, name="c1_vector_vs_probabilities_by_dataset", names=names))
    for view in ("per_column", "joint"):
        gcfig.render(save, gcfig.c1_gains(summary, view=view, name=f"c1_gains_{view}"))
    display_table("Gain over no text per arm, predictor and task family (equal-dataset summary)", summary, preview_rows=20)
    display_table("Fold-paired comparisons (A minus B) per predictor and task family", contrasts, preview_rows=20)

'Waiting for the C1 export (c1_representations.zip): nothing to show yet.'

## 4. Statistics (gated until the owner approves the framework)

Framework application `C1_strands_states` in `config/05_synthesis/analysis.yaml`. While `framework.status` is
`awaiting_owner_approval`, this section prints the gate only: nothing is read and no test is run. After
approval: Wilcoxon signed-rank with Holm adjustment within the one declared family, the Bayesian
signed-rank test with a ROPE, descriptive Friedman/critical-difference ranks, and fold-level win rates and
PAMA with dataset-cluster bootstrap intervals.

In [5]:
report_section("4. Statistics (gated)")
framework = learner_scores.run_application("C1_strands_states")
display_local(f"C1_strands_states: {framework['status']} (framework.status = {framework['framework_status']})")
display_json("Framework application C1_strands_states", learner_scores.summary_record(framework))
if framework["status"] == "compared":
    gcfig.render(save, critical_difference.cd_figures(framework))
else:
    display_local("Statistics figures are not drawn until the framework is approved and the export exists.")

'C1_strands_states: awaiting_owner_approval (framework.status = awaiting_owner_approval)'

'Statistics figures are not drawn until the framework is approved and the export exists.'

## 5. Complete report

In [6]:
finish_report(NOTEBOOK, save.summary() + "\nOnly verified Group C exports were read; no inference, training or VSC change was performed.")

C/exp10_representations

Notebook execution identity
{
  "code_sha256": "41477e70c06f4bea09495570fa55522fe62d639d69a854cd48125d8fa3d2c814",
  "git_head": "ac34e04f34066a8bc54cc08092ca38ea3125314b",
  "git_dirty": true,
  "tfm_library_pin": "4574eea0c2ce6646b5d2e9c7f53a6ef3882c368c dm-library (remotes/origin/HEAD)\n 248f137046487e0d57360abba1e060420ae54e1a tfm-library (remotes/origin/HEAD)\n 2208cf68d6351eff2cb0ddb5a9eb1d822523457b vsc-library (heads/main-8-g2208cf6)",
  "environment": {
    "python": "3.12.10",
    "numpy": "2.5.4",
    "pandas": "2.3.3",
    "PyYAML": "6.0.3",
    "matplotlib": "3.11.2",
    "pyarrow": "25.0.1"
  }
}

Waiting for export: C:\Users\U0152019\PhD Documents\Projects\5. JEVPFN\JEVPFN\output_JEVPFN\C\exp10_representations\results\group_c\c1_representations.zip is absent. No measurements are shown.


Export identity
{
  "status": "awaiting_export",
  "path": "C:\\Users\\U0152019\\PhD Documents\\Projects\\5. JEVPFN\\JEVPFN\\output_JEVPFN\\C\\exp10_representati